`exp2/g1` 의 12개 `footstep_eval_*.csv` 시도(각 목표 100 step)를 불러와
**도달 스텝 수**, **planar foothold error** $|xy|=\sqrt{e_x^2+e_y^2}$ [cm],
**yaw error** $|e_{\mathrm{yaw}}|$ [deg] 의 mean $\pm$ std 를 구한 뒤
3$\times$4 LaTeX 표를 출력합니다. 100행이면 완주, 그보다 적으면 실패입니다.
로그 stdout은 m / rad, Overleaf 표는 $|xy|$ cm, $|yaw|$ deg입니다.


In [4]:
from pathlib import Path
import csv
import numpy as np

HERE = Path("/home/yong/unitree_ws/g1_controller/log/exp2/g1")
N_TARGET = 100
NROWS, NCOLS = 3, 4
FILES = sorted(p.name for p in HERE.glob("footstep_eval_*.csv"))


def mean_std(vals):
    a = np.asarray(vals, dtype=float)
    if a.size == 0:
        return None
    if a.size == 1:
        return float(a[0]), float("nan")
    return float(a.mean()), float(a.std(ddof=1))


def load_trial(path):
    xy, yaw = [], []
    rows = []
    with path.open() as f:
        for row in csv.DictReader(f):
            xy.append(np.hypot(float(row["err_x"]), float(row["err_y"])))
            yaw.append(abs(float(row["err_yaw"])))
            rows.append(row)
    n = len(xy)
    return {
        "n": n,
        "ok": n >= N_TARGET,
        "xy_vals": xy,
        "yaw_vals": yaw,
        "xy": mean_std(xy),
        "yaw": mean_std(yaw),
        "last": rows[-1] if rows else None,
        "rows": rows,
    }


trials = []
for sid, name in enumerate(FILES, 1):
    t = load_trial(HERE / name)
    t["id"] = sid
    t["file"] = name
    trials.append(t)
assert len(trials) == NROWS * NCOLS, f"expected {NROWS * NCOLS} trials, got {len(trials)}"

for t in trials:
    status = "OK" if t["ok"] else "FAIL"
    xy, yaw = t["xy"], t["yaw"]
    print(
        f"Trial {t['id']:2d}  {t['n']:3d}/{N_TARGET}  {status:4s}  "
        f"|xy| {100*xy[0]:.2f} ± {100*xy[1]:.2f} cm   "
        f"|yaw| {np.degrees(yaw[0]):.2f} ± {np.degrees(yaw[1]):.2f} deg   {t['file']}"
    )

all_xy = [v for t in trials for v in t["xy_vals"]]
all_yaw = [v for t in trials for v in t["yaw_vals"]]
n_ok = sum(t["ok"] for t in trials)
xy_all, yaw_all = mean_std(all_xy), mean_std(all_yaw)
print(
    f"All steps pooled  n={len(all_xy)}  "
    f"|xy| {100*xy_all[0]:.2f} ± {100*xy_all[1]:.2f} cm   "
    f"|yaw| {np.degrees(yaw_all[0]):.2f} ± {np.degrees(yaw_all[1]):.2f} deg   "
    f"success {n_ok}/{len(trials)}"
)

print("\nFailed last footstep command (not in figure):")
for t in trials:
    if t["ok"] or t["last"] is None:
        continue
    tail = t["rows"][-3:]
    labels = {0: "전전", 1: "전", 2: "last"}
    offset = 3 - len(tail)
    print(f"  T{t['id']}")
    for k, r in enumerate(tail):
        tag = labels[k + offset]
        print(
            f"    {tag:4s}  step {r['step']} {r['foot']}  "
            f"cmd_x={float(r['cmd_x']):+.3f}  cmd_y={float(r['cmd_y']):+.3f}  "
            f"cmd_z={float(r['cmd_z']):+.3f}  cmd_yaw={float(r['cmd_yaw']):+.3f}"
        )


def shade(t):
    return "green!20" if t["ok"] else "red!20"


def pm(ms, scale=1.0, digits=3):
    m, s = ms
    return f"${m * scale:.{digits}f} \\pm {s * scale:.{digits}f}$"


def cell(i, t):
    if t is None:
        return r"\cellcolor{gray!10} --"
    inner = (
        r"\begin{tabular}{@{}c@{}}"
        + rf"\textbf{{T{i}}} \; {t['n']}/{N_TARGET}"
        + r" \\ "
        + pm(t["xy"], scale=100.0, digits=2)
        + r" \\ "
        + pm(t["yaw"], scale=180.0 / np.pi, digits=2)
        + r"\end{tabular}"
    )
    return rf"\cellcolor{{{shade(t)}}} {inner}"


by_id = {t["id"]: t for t in trials}
grid = [by_id.get(i) for i in range(1, NROWS * NCOLS + 1)]

rows = []
for r in range(NROWS):
    cells = " & ".join(
        cell(r * NCOLS + c + 1, grid[r * NCOLS + c]) for c in range(NCOLS)
    )
    rows.append(cells + r" \\")
body = "\n".join(rows)

latex = (
    r"% \usepackage{booktabs}" + "\n"
    r"% \usepackage[table]{xcolor}" + "\n"
    r"\begin{table}[t]" + "\n"
    r"\centering" + "\n"
    r"\caption{Twelve G1 trials of 100 commanded footsteps. "
    r"Each cell: reached steps ($n/100$), mean $\pm$ std of planar "
    r"foothold error $|xy|$ (cm, middle) and yaw error $|yaw|$ (deg, bottom). "
    r"Green: completed 100 steps; red: fell before 100.}" + "\n"
    r"\label{tab:g1-12trials}" + "\n"
    r"\begin{tabular}{" + "c" * NCOLS + "}" + "\n"
    r"\toprule" + "\n"
    + body + "\n"
    r"\bottomrule" + "\n"
    r"\end{tabular}" + "\n"
    r"\end{table}" + "\n"
)
print()
print(latex)


Trial  1  100/100  OK    |xy| 4.10 ± 2.22 cm   |yaw| 3.72 ± 3.30 deg   footstep_eval_260904_175041.csv
Trial  2  100/100  OK    |xy| 3.89 ± 1.86 cm   |yaw| 3.43 ± 2.73 deg   footstep_eval_260904_175834.csv
Trial  3  100/100  OK    |xy| 3.98 ± 2.15 cm   |yaw| 3.63 ± 3.10 deg   footstep_eval_260904_180054.csv
Trial  4  100/100  OK    |xy| 3.95 ± 2.22 cm   |yaw| 3.29 ± 2.49 deg   footstep_eval_260904_180315.csv
Trial  5  100/100  OK    |xy| 4.11 ± 2.07 cm   |yaw| 3.50 ± 3.03 deg   footstep_eval_260904_180530.csv
Trial  6  100/100  OK    |xy| 4.14 ± 2.16 cm   |yaw| 3.43 ± 2.87 deg   footstep_eval_260904_180749.csv
Trial  7  100/100  OK    |xy| 3.96 ± 2.09 cm   |yaw| 3.24 ± 2.92 deg   footstep_eval_260904_181005.csv
Trial  8  100/100  OK    |xy| 4.06 ± 2.24 cm   |yaw| 3.66 ± 3.31 deg   footstep_eval_260904_181237.csv
Trial  9  100/100  OK    |xy| 4.11 ± 2.18 cm   |yaw| 3.24 ± 2.84 deg   footstep_eval_260904_181819.csv
Trial 10  100/100  OK    |xy| 4.05 ± 2.34 cm   |yaw| 3.52 ± 3.12 deg   fo